In [10]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, lower, lit
import logging
import glob
from datetime import datetime

# Configuração de logging (continua)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- DEFINIÇÕES DE CAMINHOS ---
BASE_PATH = './data_lake'
BRONZE_PATH = os.path.join(BASE_PATH, "bronze")
SILVER_PATH = os.path.join(BASE_PATH, "silver")
# ------------------------------

def create_spark_session() -> SparkSession:
    """Inicia a sessão Spark."""
    spark = (
        SparkSession.builder
        .appName("BEES_PySpark_LoadSilver")
        .master("local[*]") # Usa todos os cores disponíveis
        .getOrCreate()
    )
    logging.info("Spark session criada com sucesso.")
    return spark

def get_latest_bronze_file(bronze_path: str) -> str:
    """Encontra o arquivo JSON mais recente na pasta Bronze."""
    list_of_files = glob.glob(os.path.join(bronze_path, '*.json'))
    if not list_of_files:
        raise FileNotFoundError(f"No JSON files found in {bronze_path}")
    latest_file = max(list_of_files, key=os.path.getctime)
    logging.info(f"Usando arquivo Bronze mais recente: {latest_file}")
    return latest_file


def transform_to_silver(spark: SparkSession, bronze_file: str):
    """
    Lê Bronze, aplica transformações e retorna o DataFrame Silver.
    Transformações: Limpeza e padronização dos campos de partição/análise.
    """
    logging.info(f"Lendo dados JSON da Bronze: {bronze_file}")
    # Lê JSON multi-linhas com schema inferido
    df_bronze = spark.read.json(bronze_file, multiLine=True)
    
    total_records = df_bronze.count()
    logging.info(f"Total de registros lidos da Bronze: {total_records}")

    # Aplica transformações (limpeza e padronização, conforme o item 5b)
    df_silver = df_bronze.withColumn("country", lower(trim(col("country")))) \
                         .withColumn("state_province", lower(trim(col("state_province")))) \
                         .withColumn("city", lower(trim(col("city")))) \
                         .withColumn("brewery_type", lower(trim(col("brewery_type"))))
                         
    # Adicionando uma coluna de controle para auditoria (melhora o Bronze)
    df_silver = df_silver.withColumn("load_timestamp", lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))

    logging.info(f"Transformação Silver concluída. Total de registros: {df_silver.count()}")
    return df_silver

def write_silver_data(df, silver_path: str):
    """Grava o DataFrame na camada Silver no formato Parquet, particionado por localização."""
    os.makedirs(silver_path, exist_ok=True)
    logging.info(f"Gravando camada Silver em: {silver_path}")
    
    # REQUISITO CRUCIAL DO SILVER LAYER: Particionar por localização 
    # Usando 'country' e 'state_province' como chaves de partição.
    df.write.mode("overwrite").partitionBy("country", "state_province").parquet(silver_path)
    
    logging.info("Camada Silver particionada gerada com sucesso.")

def run_silver_pipeline():
    spark: Optional[SparkSession] = None
    try:
        spark = create_spark_session()

        # 1. Obter arquivo Bronze mais recente
        bronze_file = get_latest_bronze_file(BRONZE_PATH)

        # 2. Transformação e Limpeza
        df_silver = transform_to_silver(spark, bronze_file)

        # 3. Escrita Silver (Particionada)
        write_silver_data(df_silver, SILVER_PATH)

        logging.info("Pipeline Bronze → Silver concluído com sucesso!")

    except Exception as e:
        logging.error(f"Erro ao gerar Silver: {e}")
        import traceback
        logging.error(traceback.format_exc())
    finally:
        if spark:
            spark.stop()


In [11]:
run_silver_pipeline()

2025-10-21 00:27:12,583 - INFO - Spark session criada com sucesso.
2025-10-21 00:27:12,596 - INFO - Usando arquivo Bronze mais recente: ./data_lake/bronze/breweries_20251021_001816.json
2025-10-21 00:27:12,599 - INFO - Lendo dados JSON da Bronze: ./data_lake/bronze/breweries_20251021_001816.json
2025-10-21 00:27:14,839 - INFO - Total de registros lidos da Bronze: 8923
2025-10-21 00:27:15,596 - INFO - Transformação Silver concluída. Total de registros: 8923
2025-10-21 00:27:15,609 - INFO - Gravando camada Silver em: ./data_lake/silver
2025-10-21 00:28:01,987 - INFO - Camada Silver particionada gerada com sucesso.
2025-10-21 00:28:01,989 - INFO - Pipeline Bronze → Silver concluído com sucesso!
